# 06. 해상도 개선 EDA

**목적**: 현재 800×800 Letterbox 기준에서 알약 BBox 크기 분포를 분석하고,  
해상도를 높였을 때 탐지 성능이 향상될 근거를 데이터로 확인합니다.

**분석 항목**:
1. BBox 픽셀 크기 분포 (800px 기준)
2. 해상도별 BBox 크기 시뮬레이션 (800 / 1024 / 1280)
3. COCO 기준 소형(small) 객체 비율
4. 클래스별 평균 BBox 크기 → 소수 클래스와 크기 연관성
5. 해상도 선택 근거 정리

In [ ]:
# ============================================================
# [Cell 0] 환경 세팅
# ============================================================
import os, sys

# Colab
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = '/content/pill_detection_project'
    if not os.path.exists(REPO_DIR):
        os.system(f'git clone https://github.com/EuijeongHan/pill_detection_project.git {REPO_DIR}')
        os.system(f'cd {REPO_DIR} && git checkout detr-resolution')
    sys.path.insert(0, REPO_DIR)
    BASE_DIR = '/content/drive/MyDrive/data/초급_프로젝트/dataset'
# 로컬
else:
    BASE_DIR = '../data'

TRAIN_JSON = f'{BASE_DIR}/train_letterbox.json'
VAL_JSON   = f'{BASE_DIR}/val_letterbox.json'

print(f'✅ BASE_DIR    : {BASE_DIR}')
print(f'✅ TRAIN_JSON  : {TRAIN_JSON}')

In [ ]:
# ============================================================
# [Cell 1] 데이터 로드
# ============================================================
import json
import pandas as pd
import numpy as np

with open(TRAIN_JSON, 'r') as f:
    train = json.load(f)
with open(VAL_JSON, 'r') as f:
    val = json.load(f)

# category_id → 이름 매핑
cat_names = {c['id']: c['name'] for c in train['categories']}

# 전체 annotation DataFrame
anns = train['annotations'] + val['annotations']
df = pd.DataFrame([
    {
        'category_id': a['category_id'],
        'class_name':  cat_names.get(a['category_id'], str(a['category_id'])),
        'x': a['bbox'][0], 'y': a['bbox'][1],
        'w': a['bbox'][2], 'h': a['bbox'][3],
        'area': a['bbox'][2] * a['bbox'][3],
    }
    for a in anns
])

print(f'전체 annotation 수 : {len(df):,}개')
print(f'클래스 수          : {df["category_id"].nunique()}종')
print(f'\nBBox 크기 요약 (800px 기준)')
print(df[['w','h','area']].describe().round(1))

In [ ]:
# ============================================================
# [Cell 2] BBox 크기 분포 시각화 (800px 기준)
# ============================================================
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'AppleGothic'  # Mac 한글
# Colab: matplotlib.rcParams['font.family'] = 'NanumGothic'

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('BBox 크기 분포 (800×800 Letterbox 기준)', fontsize=14)

axes[0].hist(df['w'], bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(df['w'].median(), color='red', linestyle='--', label=f'중앙값 {df["w"].median():.0f}px')
axes[0].set_title('BBox 너비 (w)')
axes[0].set_xlabel('픽셀'); axes[0].legend()

axes[1].hist(df['h'], bins=50, color='darkorange', edgecolor='white')
axes[1].axvline(df['h'].median(), color='red', linestyle='--', label=f'중앙값 {df["h"].median():.0f}px')
axes[1].set_title('BBox 높이 (h)')
axes[1].set_xlabel('픽셀'); axes[1].legend()

axes[2].scatter(df['w'], df['h'], alpha=0.2, s=5, color='purple')
axes[2].set_title('너비 vs 높이 산점도')
axes[2].set_xlabel('w (px)'); axes[2].set_ylabel('h (px)')

plt.tight_layout()
plt.show()

print(f'\n너비  중앙값: {df["w"].median():.1f}px  /  평균: {df["w"].mean():.1f}px')
print(f'높이  중앙값: {df["h"].median():.1f}px  /  평균: {df["h"].mean():.1f}px')

In [ ]:
# ============================================================
# [Cell 3] COCO 기준 소형 객체 비율
# ============================================================
# COCO 정의: small  = area < 32^2 = 1024
#             medium = 32^2 ~ 96^2
#             large  = area > 96^2 = 9216

small  = (df['area'] < 32**2).sum()
medium = ((df['area'] >= 32**2) & (df['area'] < 96**2)).sum()
large  = (df['area'] >= 96**2).sum()
total  = len(df)

print('=== COCO 기준 객체 크기 분류 (800px 기준) ===')
print(f'small  (< 32²)     : {small:>5,}개  ({small/total*100:.1f}%)')
print(f'medium (32²~96²)   : {medium:>5,}개  ({medium/total*100:.1f}%)')
print(f'large  (> 96²)     : {large:>5,}개  ({large/total*100:.1f}%)')

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['small\n(<32²)', 'medium\n(32²~96²)', 'large\n(>96²)'],
       [small, medium, large],
       color=['#e74c3c','#f39c12','#2ecc71'])
ax.set_title('COCO 기준 객체 크기 분포 (800px)')
ax.set_ylabel('annotation 수')
for i, v in enumerate([small, medium, large]):
    ax.text(i, v + 10, f'{v/total*100:.1f}%', ha='center')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# [Cell 4] 해상도별 BBox 크기 시뮬레이션
# ============================================================
# Letterbox는 long-side 기준 스케일이므로
# 해상도가 올라가면 BBox도 동일 비율로 커집니다.

resolutions = [800, 1024, 1280]
scale_factors = {r: r / 800 for r in resolutions}

print('=== 해상도별 BBox 크기 시뮬레이션 ===')
print(f'{"":20} {"800px":>10} {"1024px":>10} {"1280px":>10}')
print('-' * 52)

for stat, label in [(df['w'].median(), 'w 중앙값'),
                    (df['w'].mean(),   'w 평균'),
                    (df['h'].median(), 'h 중앙값'),
                    (df['h'].mean(),   'h 평균')]:
    vals = [f'{stat * scale_factors[r]:.1f}px' for r in resolutions]
    print(f'{label:<20} {vals[0]:>10} {vals[1]:>10} {vals[2]:>10}')

print()

# 해상도별 small 객체 비율 변화
print('=== 해상도별 small 객체 비율 변화 ===')
for r in resolutions:
    scale = r / 800
    scaled_area = df['area'] * (scale ** 2)
    s = (scaled_area < 32**2).sum()
    print(f'{r}px : small 비율 {s/total*100:.1f}%  ({s:,}개)')

In [ ]:
# ============================================================
# [Cell 5] 해상도별 BBox 분포 비교 시각화
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
fig.suptitle('해상도별 BBox 너비 분포 시뮬레이션', fontsize=13)

colors = ['#3498db', '#e67e22', '#2ecc71']

for ax, r, color in zip(axes, resolutions, colors):
    scale  = r / 800
    scaled_w = df['w'] * scale
    ax.hist(scaled_w, bins=50, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(scaled_w.median(), color='red', linestyle='--',
               label=f'중앙값 {scaled_w.median():.0f}px')
    ax.axvline(32, color='black', linestyle=':', alpha=0.6, label='COCO small 기준(32px)')
    ax.set_title(f'{r}×{r}')
    ax.set_xlabel('BBox 너비 (px)')
    ax.legend(fontsize=8)

axes[0].set_ylabel('annotation 수')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# [Cell 6] 클래스별 평균 BBox 크기 (하위 20개 — 가장 작은 알약)
# ============================================================
class_size = (
    df.groupby('class_name')[['w', 'h', 'area']]
    .mean()
    .assign(short_side=lambda d: d[['w','h']].min(axis=1))
    .sort_values('short_side')
)

bottom20 = class_size.head(20)

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(bottom20.index, bottom20['short_side'], color='#e74c3c')
ax.axvline(32, color='black', linestyle='--', label='COCO small 기준 (32px)')
ax.set_xlabel('평균 단변 길이 (px, 800px 기준)')
ax.set_title('BBox 가장 작은 클래스 TOP 20 (800px 기준)')
ax.legend()
plt.tight_layout()
plt.show()

print('\n가장 작은 클래스 TOP 10 (800px 기준):')
print(bottom20[['w','h','short_side']].head(10).round(1).to_string())

In [ ]:
# ============================================================
# [Cell 7] 해상도 선택 근거 요약
# ============================================================
print('=' * 60)
print('해상도 실험 계획 요약')
print('=' * 60)

for r in resolutions:
    scale      = r / 800
    med_w      = df['w'].median() * scale
    med_h      = df['h'].median() * scale
    small_cnt  = (df['area'] * scale**2 < 32**2).sum()
    small_pct  = small_cnt / total * 100
    print(f'\n[{r}×{r}]')
    print(f'  BBox 중앙값   : {med_w:.1f} × {med_h:.1f} px')
    print(f'  small 비율    : {small_pct:.1f}%  ({small_cnt:,}개)')
    print(f'  VRAM 영향     : batch_size 기준 ~{max(1, int(4 / scale**2))}~{max(1, int(8 / scale**2))} 권장')

print()
print('권장: 1024px — small 비율 감소 + VRAM 부담 균형')